In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from scipy import stats
from numpy.linalg import inv
import warnings
warnings.filterwarnings('ignore')

In [2]:
from google.colab import drive
drive.mount('/content/drive')
trees = pd.read_csv("/content/drive/MyDrive/DATA 311 Redwood Analytics/Project 3/biomass_model_dataset.csv")
species = pd.read_csv("/content/drive/MyDrive/DATA 311 Redwood Analytics/Project 3/FIATreeSpeciesCode.csv")

Mounted at /content/drive


In [3]:
trees.head()

,SPCD,DO_BH,HT_TOT,TT_DW_CRM
0,12,7.5,45.2,178.86
1,12,9.3,62.7,323.46
2,12,9.7,56.7,342.47
3,12,11.4,66.1,499.35
4,12,6.4,51.8,113.92


In [4]:
species.head()

,SPCD,COMMON_NAME,SCI_NAME,SPGRPCD.E,SPGRPCD.W,MAJGRP,OCC.NC,OCC.NE,OCC.PNW,OCC.RM,OCC.SO
0,10,fir spp.,Abies spp,6,12,2,X,X,-,-,X
1,11,Pacific silver fir,Abies amabilis,9,12,2,-,-,X,-,-
2,12,balsam fir,Abies balsamea,6,12,2,X,X,-,-,X
3,14,Santa Lucia or bristlecone fir,Abies bracteata,9,12,2,-,-,X,-,-
4,15,white fir,Abies concolor,9,12,2,X,-,X,X,X


In [5]:
species_codes = species[["SPCD","COMMON_NAME"]]
combined_df = trees.merge(species_codes,on='SPCD', how='left')
combined_df.head()

,SPCD,DO_BH,HT_TOT,TT_DW_CRM,COMMON_NAME
0,12,7.5,45.2,178.86,balsam fir
1,12,9.3,62.7,323.46,balsam fir
2,12,9.7,56.7,342.47,balsam fir
3,12,11.4,66.1,499.35,balsam fir
4,12,6.4,51.8,113.92,balsam fir


In [6]:
sum(combined_df["COMMON_NAME"].isnull())

94

In [7]:
combined_df = combined_df.dropna(subset = "COMMON_NAME")

In [8]:
counts = combined_df["COMMON_NAME"].value_counts()
frequent_values = counts[counts >= 100].index
df_filtered = combined_df[combined_df["COMMON_NAME"].isin(frequent_values)]
df_filtered

,SPCD,DO_BH,HT_TOT,TT_DW_CRM,COMMON_NAME
0,12,7.5,45.2,178.86,balsam fir
1,12,9.3,62.7,323.46,balsam fir
2,12,9.7,56.7,342.47,balsam fir
3,12,11.4,66.1,499.35,balsam fir
4,12,6.4,51.8,113.92,balsam fir
...,...,...,...,...,...
137035,691,3.7,25.4,40.76,water tupelo
137036,691,3.2,28.8,28.44,water tupelo
137037,691,4.7,31.4,73.78,water tupelo
137038,691,4.4,26.9,62.64,water tupelo


In [9]:
df_filtered["COMMON_NAME"].value_counts()

,count
COMMON_NAME,
loblolly pine,20140
slash pine,8006
Douglas-fir,7322
ponderosa pine,5609
shortleaf pine,5599
...,...
sweet birch,135
sugarberry,120
pecan,120


In [10]:
train = []
test = []
selection = []
tree = []
for i in range(len(df_filtered)):
  #ntrain = df_filtered["COMMON_NAME"].value_counts()[0] * 0.80
  #nst = df_filtered["COMMON_NAME"].value_counts()[0] - ntrain
  #nselection = nst/2
  #ntest = nselection
  if (df_filtered["COMMON_NAME"].iloc[i] == "loblolly pine"):
    tree.append(df_filtered.iloc[i])


In [11]:
##Testing data splits for just one species code
#make dataframe with certain common name
tree = df_filtered.loc[df_filtered['COMMON_NAME'] == "loblolly pine"]

#find last index for tarining, test, and selection index
train_index = int(tree["COMMON_NAME"].value_counts()[0] * 0.80)
test_index = int(tree["COMMON_NAME"].value_counts()[0] * 0.90)
selection_index = int(tree["COMMON_NAME"].value_counts()[0])

#put in new dataframes
train = pd.DataFrame(tree.iloc[:train_index])
test = pd.DataFrame(tree.iloc[train_index:test_index])
selection = pd.DataFrame(tree.iloc[test_index:selection_index])

train.to_csv('/content/drive/MyDrive/one_species_train.csv',index=False)
test.to_csv('/content/drive/MyDrive/one_species_test.csv',index=False)
selection.to_csv('/content/drive/MyDrive/one_species_selection.csv',index=False)


In [12]:
train

,SPCD,DO_BH,HT_TOT,TT_DW_CRM,COMMON_NAME
1312,131,2.1,14.3,8.15,loblolly pine
1313,131,2.9,13.9,17.75,loblolly pine
1314,131,1.7,11.5,4.91,loblolly pine
1315,131,1.9,12.3,6.41,loblolly pine
1316,131,1.9,12.2,6.41,loblolly pine
...,...,...,...,...,...
122287,131,10.4,72.2,669.67,loblolly pine
122288,131,13.1,70.5,1045.60,loblolly pine
122289,131,14.4,64.7,1165.20,loblolly pine
122290,131,4.8,29.6,60.16,loblolly pine


In [13]:
test

,SPCD,DO_BH,HT_TOT,TT_DW_CRM,COMMON_NAME
122292,131,5.1,20.7,20.74,loblolly pine
122293,131,7.0,27.6,98.93,loblolly pine
122294,131,6.1,32.9,85.09,loblolly pine
122295,131,6.4,33.0,97.21,loblolly pine
122296,131,7.2,36.5,147.37,loblolly pine
...,...,...,...,...,...
126027,131,6.5,50.5,167.28,loblolly pine
126028,131,3.9,39.6,36.36,loblolly pine
126029,131,6.7,46.0,161.52,loblolly pine
126030,131,4.6,35.6,54.26,loblolly pine


In [14]:
selection

,SPCD,DO_BH,HT_TOT,TT_DW_CRM,COMMON_NAME
126032,131,5.0,27.5,34.15,loblolly pine
126033,131,5.4,44.2,89.64,loblolly pine
126034,131,2.2,16.7,9.11,loblolly pine
126035,131,3.0,23.3,19.26,loblolly pine
126036,131,7.2,63.1,269.50,loblolly pine
...,...,...,...,...,...
136966,131,9.7,57.2,460.06,loblolly pine
136967,131,9.3,55.3,407.37,loblolly pine
136968,131,9.1,67.3,474.54,loblolly pine
136969,131,11.0,77.3,803.18,loblolly pine


In [15]:
##Spliting the full dataset to be 80% of each species in training, 10% of each species in selection, and 10% of each species in testing
train_list=[]
test_list=[]
selection_list=[]

for species_name in df_filtered['COMMON_NAME'].unique():
    tree = df_filtered.loc[df_filtered['COMMON_NAME'] == species_name]
    #find last index for tarining, test, and selection index
    train_index = int(tree["COMMON_NAME"].value_counts()[0] * 0.80)
    test_index = int(tree["COMMON_NAME"].value_counts()[0] * 0.90)
    selection_index = int(tree["COMMON_NAME"].value_counts()[0])

    #put in new dataframes
    train = pd.DataFrame(tree.iloc[:train_index])
    test = pd.DataFrame(tree.iloc[train_index:test_index])
    selection = pd.DataFrame(tree.iloc[test_index:selection_index])

    train_list.append(train)
    test_list.append(test)
    selection_list.append(selection)

# Combine all species
train_df = pd.concat(train_list, ignore_index=True)
test_df = pd.concat(test_list, ignore_index=True)
selection_df = pd.concat(selection_list, ignore_index=True)

train_df.to_csv('/content/drive/MyDrive/training_data.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/testing_data.csv', index=False)
selection_df.to_csv('/content/drive/MyDrive/selection_data.csv', index=False)


In [16]:
train_df

,SPCD,DO_BH,HT_TOT,TT_DW_CRM,COMMON_NAME
0,12,7.5,45.2,178.86,balsam fir
1,12,9.3,62.7,323.46,balsam fir
2,12,9.7,56.7,342.47,balsam fir
3,12,11.4,66.1,499.35,balsam fir
4,12,6.4,51.8,113.92,balsam fir
...,...,...,...,...,...
107641,461,17.5,87.4,2042.09,sugarberry
107642,461,10.0,66.5,565.80,sugarberry
107643,461,6.2,54.2,191.04,sugarberry
107644,461,16.0,82.5,1632.64,sugarberry


In [17]:
test_df

,SPCD,DO_BH,HT_TOT,TT_DW_CRM,COMMON_NAME
0,12,1.7,12.1,4.37,balsam fir
1,12,10.7,77.4,485.56,balsam fir
2,12,7.4,50.5,168.61,balsam fir
3,12,3.9,33.5,32.99,balsam fir
4,12,5.8,39.7,87.47,balsam fir
...,...,...,...,...,...
13448,461,13.3,73.9,1043.74,sugarberry
13449,461,5.4,40.6,109.32,sugarberry
13450,461,1.0,16.6,1.55,sugarberry
13451,461,1.1,10.9,1.95,sugarberry


In [18]:
selection_df

,SPCD,DO_BH,HT_TOT,TT_DW_CRM,COMMON_NAME
0,12,6.0,33.4,92.86,balsam fir
1,12,9.2,47.8,321.11,balsam fir
2,12,8.3,52.6,256.71,balsam fir
3,12,11.7,55.0,558.90,balsam fir
4,12,3.4,21.7,23.55,balsam fir
...,...,...,...,...,...
13498,461,21.9,94.3,3365.12,sugarberry
13499,461,17.6,85.0,2009.27,sugarberry
13500,461,9.0,54.8,383.58,sugarberry
13501,461,12.8,64.7,857.65,sugarberry
